In [ ]:
%%capture
!pip install --no-deps unsloth unsloth_zoo bitsandbytes accelerate peft trl
!pip install --no-deps "torchao>=0.16.0"
!pip install transformers>=4.48.0

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = 4096,
    load_in_4bit = True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
import urllib.request

# Download your data
urllib.request.urlretrieve("https://raw.githubusercontent.com/Youmei295/DeAIze-model-source-code/4a8585904c39891d9072edf752883fe83dfb211b/dataset.json", "dataset.json")
dataset = load_dataset("json", data_files="dataset.json", split="train")

# Set the template
tokenizer = get_chat_template(tokenizer, chat_template="qwen2.5")

def formatting_func(examples):
    texts = []
    for inp, tar in zip(examples["input"], examples["target"]):
        messages = [
            {"role": "system", "content": "You are a professional Vietnamese editor. Rewrite the text to be natural, human-sounding, and concise. Remove robotic AI-isms and repetitive transitions."},
            {"role": "user", "content": inp},
            {"role": "assistant", "content": tar},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 1. Define your Training Arguments
# We use a lower max_steps since we are resuming from a checkpoint
training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60, # Reduced from 100 since we are refining an existing model
    learning_rate = 5e-5, # Lower learning rate for incremental tuning
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",
)

# 2. Initialize the Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 4096,
    dataset_num_proc = 2,
    packing = False,
    args = training_args,
)

# 3. Train the model
# The trainer will automatically handle the weights we loaded in the previous cell.
# If we loaded the base model, it trains from scratch. 
# If we loaded your previous fine-tuned model, it resumes from that point.
trainer.train()

In [ ]:
# Save locally first, then push
model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")
model.push_to_hub_gguf("Youmei295/deAIze", tokenizer, quantization_method="q4_k_m", token=hf_token)